In [ ]:
import faultier
import serial
import time

from tqdm.notebook import trange
ft = faultier.Faultier()
lp = faultier.LiveMarkerPlotOld()
# pd = faultier.GlitchDataCollection()

ser = serial.Serial(ft.get_serial_path(), baudrate=115200)
ser.timeout = 0.5
ft.configure_glitcher(
    power_cycle_output = faultier.OUT_MUX0,
    power_cycle_length = 300000,
    # trigger_source = faultier.TRIGGER_IN_EXT0,
    # trigger_type = faultier.TRIGGER_PULSE_POSITIVE,
    trigger_source = faultier.TRIGGER_IN_EXT1,
    trigger_type = faultier.TRIGGER_RISING_EDGE,
    glitch_output = faultier.OUT_CROWBAR
)

# dry run
ft.power_cycle()
print(ser.read(3))

success = []
fails = []
detected = []

for d in trange(0, 100):
    for p in range(200, 400):
        if ser.in_waiting:
            ser.read(ser.in_waiting)
        ft.glitch(delay=d, pulse=p)
        data = ser.read(6)

    # print(data)
    try:
        if b"Q" in data:
            print(f"Successful glitch - Delay: {d} Pulse: {p}")
            print(data)
            success.append([d, p])
            lp.update([success, detected])
        elif b"E" in data:
            print(f"Expected glitch at target node - Delay: {d} Pulse: {p}")
            print(data)
            success.append([d, p])
            lp.update([success, detected])
        else:
            fails.append([d, p])
    except:
        fails.append([d, p])